### Import Libraries and Load Survey Data

In [312]:
import pandas as pd
import re
import requests
import numpy as np

df = pd.read_csv("../raw_data/simulated_commuter_survey_5yrs_messy_100.csv")
df.head(10)

,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est
0,F_2021-22_00104,2021-22,Faculty,Blue Lake,955,South Rail St & Hatchery Rd,9.63,9.0,36.0,0.444444,0.000000,0.111111,0.111111,0.111111,0.222222,Drive alone,False,False,0.580547,325.0
1,F_2024-25_00271,2024-25,Faculty,Fortuna,95540.0,Main St & Kenmar Rd,20.56,9.0,37.0,0.555556,-0.100000,0.000000,0.111111,-0.050000,0.111111,Drive alone,False,False,0.987977,348.7
2,S_2024-25_00154,2024-25,Staff/Admin,Trinidad,95570.0,Patrick Point Dr & Trinity St,11.82,10.0,48.0,0.800000,0.100000,0.000000,0.000000,0.000000,0.100000,Drive alone,False,False,2.982530,486.0
3,S_2024-25_00197,2024-25,Student,Eureka,95501.0,I St & K St,8.61,10.0,32.0,0.500000,0.100000,0.100000,0.200000,0.000000,0.100000,Drive alone,False,False,2.734667,315.0
4,S_2022-23_00708,2022-23,Student,McKinleyville,95519.0,Murray Rd & Bates Rd,13.89,10.0,32.0,0.400000,0.100000,0.100000,0.400000,0.000000,0.000000,Drive alone,False,True,4.611657,332.2
5,F_2020-21_00048,2020-21,Faculty,Arcata,95521.0,14th St & Alliance Rd,0.49,6.0,34.0,0.166667,0.000000,0.000000,0.166667,0.166667,0.500000,Telecommute,False,True,0.036916,191.9
6,S_2023-24_00496,2023-24,Student,NaN,invalid,Trinity St & Edwards St,10.44,10.0,32.0,0.300000,0.300000,0.100000,0.300000,0.000000,0.000000,Drive alone,False,False,2.715473,326.3
7,S_2022-23_00188,2022-23,Student,Eureka,95501.0,K St & Harris St,10.20,10.0,30.0,0.300000,0.100000,0.200000,0.200000,0.200000,0.000000,Drive alone,False,False,2.190158,297.2
8,S_2021-22_00554,2021-22,Student,Arcata,95521.0,Union St & H St,1.37,9.0,30.0,0.800000,-0.111111,0.500000,0.555556,0.111111,0.111111,Walk,False,False,0.350547,257.8
9,S_2020-21_00177,2020-21,Student,Arcata,95521.0,Samoa Blvd & G St,2.37,5.0,28.0,0.800000,0.000000,0.500000,0.400000,0.000000,0.200000,Walk,False,False,0.123752,133.6


### Print shape and info of DataFrame

In [313]:
print(df.shape)
print(df.info())
#print(df.describe())

(100, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   respondent_id             100 non-null    object 
 1   year                      100 non-null    object 
 2   role                      100 non-null    object 
 3   city                      97 non-null     object 
 4   zip                       97 non-null     object 
 5   nearest_intersection      100 non-null    object 
 6   distance_to_campus_miles  98 non-null     float64
 7   trips_per_week            98 non-null     float64
 8   weeks_per_year            100 non-null    float64
 9   share_drive               100 non-null    float64
 10  share_carpool             100 non-null    float64
 11  share_bus                 100 non-null    float64
 12  share_walk                100 non-null    float64
 13  share_bike                100 non-null    float64
 14  s

### Convert weeks_per_year to numeric, round mtcde_est to 2 decimal places, strip first 4 year characters to get start year, convert start_year to numeric, round and assert share_xyz columns to 2 decimal places and add to 100%

In [314]:
df['weeks_per_year'] = df['weeks_per_year'].astype(int)

In [315]:
df['year'] = df['year'].str[:4].astype(int)

In [316]:
df['mtcde_est'] = df['mtcde_est'].round(2)

In [317]:
df['total_trips_est'] = df['total_trips_est'].round(0).astype(int)

In [318]:
columns_to_round = ['share_drive', 'share_carpool', 'share_bus', 'share_walk', 'share_bike', 'share_tele']
df[columns_to_round] = df[columns_to_round].clip(lower=0)
df[columns_to_round] = df[columns_to_round].round(2)
df[columns_to_round] = df[columns_to_round].div(df[columns_to_round].sum(axis=1), axis=0).fillna(0).round(2)
df.head()

,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est
0,F_2021-22_00104,2021,Faculty,Blue Lake,955,South Rail St & Hatchery Rd,9.63,9.0,36,0.44,0.0,0.11,0.11,0.11,0.22,Drive alone,False,False,0.58,325
1,F_2024-25_00271,2024,Faculty,Fortuna,95540.0,Main St & Kenmar Rd,20.56,9.0,37,0.72,0.0,0.00,0.14,0.00,0.14,Drive alone,False,False,0.99,349
2,S_2024-25_00154,2024,Staff/Admin,Trinidad,95570.0,Patrick Point Dr & Trinity St,11.82,10.0,48,0.80,0.1,0.00,0.00,0.00,0.10,Drive alone,False,False,2.98,486
3,S_2024-25_00197,2024,Student,Eureka,95501.0,I St & K St,8.61,10.0,32,0.50,0.1,0.10,0.20,0.00,0.10,Drive alone,False,False,2.73,315
4,S_2022-23_00708,2022,Student,McKinleyville,95519.0,Murray Rd & Bates Rd,13.89,10.0,32,0.40,0.1,0.10,0.40,0.00,0.00,Drive alone,False,True,4.61,332


### Check for Null Cities, Use Geocoding on nearest_intersection to Fill Later

In [319]:
null_city = df[df['city'].notna() == False]
print([id for id in null_city.index])
null_city.head()

[6, 19, 74]


,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est
6,S_2023-24_00496,2023,Student,NaN,invalid,Trinity St & Edwards St,10.44,10.0,32,0.30,0.30,0.10,0.30,0.00,0.00,Drive alone,False,False,2.72,326
19,S_2020-21_00453,2020,Student,NaN,95519.0,Bates Rd & Central Ave,7.23,-10.0,52,0.17,0.17,0.00,0.17,0.17,0.33,Telecommute,False,False,0.65,180
74,S_2024-25_00452,2024,Student,NaN,95501.0,Harris St & I St,9.38,10.0,31,0.44,0.06,0.28,0.11,0.06,0.06,Drive alone,False,True,3.05,319


### Check for Null Zipcodes, Use Geocoding on nearest_intersection to Fill Later

In [320]:
null_zip = df[df['zip'].notna() == False]
print([id for id in null_zip.index])
null_zip.head()

[33, 36, 68]


,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est
33,F_2024-25_00046,2024,Faculty,MCKINLEYVILLE,NaN,Central Ave & Murray Rd,8.77,10.0,37,0.60,0.10,0.00,0.10,0.10,0.10,drive alone,False,True,0.75,357
36,S_2020-21_00101,2020,Student,Blue Lake,NaN,South Rail St & Greenwood Ave,7.08,-10.0,52,0.69,0.00,0.00,0.15,0.00,0.15,Telecommute,False,False,0.44,165
68,S_2023-24_00353,2023,Student,Fortuna,NaN,Newburg Rd & Kenmar Rd,25.11,10.0,33,0.44,0.06,0.28,0.11,0.06,0.06,Drive alone,False,False,3.92,326


In [321]:
null_zip.loc[:, 'zip'] = '00000'
df.update(null_zip)
df.loc[df.zip == 'invalid', 'zip'] = '00000'
df['zip'] = df['zip'].astype(str).str.ljust(5, fillchar='0').astype(float).astype(int)
df.head()

,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est
0,F_2021-22_00104,2021,Faculty,Blue Lake,95500,South Rail St & Hatchery Rd,9.63,9.0,36,0.44,0.0,0.11,0.11,0.11,0.22,Drive alone,False,False,0.58,325
1,F_2024-25_00271,2024,Faculty,Fortuna,95540,Main St & Kenmar Rd,20.56,9.0,37,0.72,0.0,0.00,0.14,0.00,0.14,Drive alone,False,False,0.99,349
2,S_2024-25_00154,2024,Staff/Admin,Trinidad,95570,Patrick Point Dr & Trinity St,11.82,10.0,48,0.80,0.1,0.00,0.00,0.00,0.10,Drive alone,False,False,2.98,486
3,S_2024-25_00197,2024,Student,Eureka,95501,I St & K St,8.61,10.0,32,0.50,0.1,0.10,0.20,0.00,0.10,Drive alone,False,False,2.73,315
4,S_2022-23_00708,2022,Student,McKinleyville,95519,Murray Rd & Bates Rd,13.89,10.0,32,0.40,0.1,0.10,0.40,0.00,0.00,Drive alone,False,True,4.61,332


In [328]:
df.loc[df.zip == 955, 'zip'] = 95500

In [329]:
df.zip.unique()

array([95500, 95540, 95570, 95501, 95519, 95521,     0, 95525])

### Check for Null Distances to Campus, Use Geocoding on nearest_intersection to Fill Later

In [330]:
null_dist_campus = df[df['distance_to_campus_miles'].notna() == False]
print([id for id in null_dist_campus.index])
null_dist_campus.head()

[84, 92]


,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est
84,S_2020-21_00346,2020,Student,Trinidad,95570,Patrick Point Dr & Main St,NaN,5.0,26,0.38,0.0,0.24,0.1,0.1,0.19,Telecommute,True,False,0.32,136
92,S_2022-23_00663,2022,Student,Blue Lake,95500,Hatchery Rd & Greenwood Ave,NaN,10.0,31,0.30,0.0,0.20,0.3,0.1,0.10,Drive alone,False,False,2.44,306


### Check for Null Trips per Week, Use Average to Fill Later

In [331]:
null_trips_perweek = df[df['trips_per_week'].notna() == False]
print([id for id in null_trips_perweek.index])
null_trips_perweek.head()

[66, 95]


,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est
66,S_2024-25_00011,2024,Student,Arcata,95521,14th St & Union St,-5.00,NaN,32,0.45,0.09,0.18,0.18,0.09,0.0,teleporting,True,True,2.01,346
95,S_2022-23_00158,2022,Student,EUREKA,95501,K St & 5th St,9.54,NaN,32,0.40,0.00,0.10,0.20,0.20,0.1,drive alone,False,False,2.87,317


In [332]:
df['trips_per_week'] = df['trips_per_week'].fillna(df['trips_per_week'].mean()).astype(int)
df.head()

,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est
0,F_2021-22_00104,2021,Faculty,Blue Lake,95500,South Rail St & Hatchery Rd,9.63,9,36,0.44,0.0,0.11,0.11,0.11,0.22,Drive alone,False,False,0.58,325
1,F_2024-25_00271,2024,Faculty,Fortuna,95540,Main St & Kenmar Rd,20.56,9,37,0.72,0.0,0.00,0.14,0.00,0.14,Drive alone,False,False,0.99,349
2,S_2024-25_00154,2024,Staff/Admin,Trinidad,95570,Patrick Point Dr & Trinity St,11.82,10,48,0.80,0.1,0.00,0.00,0.00,0.10,Drive alone,False,False,2.98,486
3,S_2024-25_00197,2024,Student,Eureka,95501,I St & K St,8.61,10,32,0.50,0.1,0.10,0.20,0.00,0.10,Drive alone,False,False,2.73,315
4,S_2022-23_00708,2022,Student,McKinleyville,95519,Murray Rd & Bates Rd,13.89,10,32,0.40,0.1,0.10,0.40,0.00,0.00,Drive alone,False,True,4.61,332


### Check for Null Mode Primary, won't matter since we provide data for all modes by default on dashboard

In [333]:
null_mode_primary = df[df['mode_primary'].notna() == False]
print([id for id in null_mode_primary.index])
null_mode_primary.head()

[78, 85, 87]


,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est
78,S_2023-24_00239,2023,Staff/Admin,ARCATA,95521,Samoa Blvd & G St,5.67,10,48,0.70,0.20,0.00,0.00,0.00,0.10,NaN,False,False,1.41,502
85,S_2022-23_00488,2022,Student,McKinleyville,95519,Central Ave & Bates Rd,11.62,11,31,0.46,0.05,0.29,0.10,0.05,0.05,NaN,False,False,4.08,347
87,F_2024-25_00279,2024,Faculty,BLUE LAKE,95525,Greenwood Ave & Chartin Rd,9.93,9,39,0.56,0.11,0.00,0.11,0.11,0.11,NaN,False,True,0.74,339


In [334]:
df.loc[df['mode_primary'] == 'drive alone', 'mode_primary'] = 'Drive Alone'
df.loc[df['mode_primary'] == 'Drive alone', 'mode_primary'] = 'Drive Alone'
df.loc[df['mode_primary'] == 'telecommute', 'mode_primary'] = 'Telecommute'
df.loc[df['mode_primary'] == 'teleporting', 'mode_primary'] = 'Telecommute'
df.loc[df['mode_primary'].isna(), 'mode_primary'] = 'No Answer'

In [335]:
df.mode_primary.unique()

array(['Drive Alone', 'Telecommute', 'Walk', 'No Answer'], dtype=object)

### Check dataframe head and info again

In [336]:
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   respondent_id             100 non-null    object 
 1   year                      100 non-null    int64  
 2   role                      100 non-null    object 
 3   city                      97 non-null     object 
 4   zip                       100 non-null    int64  
 5   nearest_intersection      100 non-null    object 
 6   distance_to_campus_miles  98 non-null     float64
 7   trips_per_week            100 non-null    int64  
 8   weeks_per_year            100 non-null    int64  
 9   share_drive               100 non-null    float64
 10  share_carpool             100 non-null    float64
 11  share_bus                 100 non-null    float64
 12  share_walk                100 non-null    float64
 13  share_bike                100 non-null    float64
 14  share_tele 

,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est
0,F_2021-22_00104,2021,Faculty,Blue Lake,95500,South Rail St & Hatchery Rd,9.63,9,36,0.44,0.0,0.11,0.11,0.11,0.22,Drive Alone,False,False,0.58,325
1,F_2024-25_00271,2024,Faculty,Fortuna,95540,Main St & Kenmar Rd,20.56,9,37,0.72,0.0,0.00,0.14,0.00,0.14,Drive Alone,False,False,0.99,349
2,S_2024-25_00154,2024,Staff/Admin,Trinidad,95570,Patrick Point Dr & Trinity St,11.82,10,48,0.80,0.1,0.00,0.00,0.00,0.10,Drive Alone,False,False,2.98,486
3,S_2024-25_00197,2024,Student,Eureka,95501,I St & K St,8.61,10,32,0.50,0.1,0.10,0.20,0.00,0.10,Drive Alone,False,False,2.73,315
4,S_2022-23_00708,2022,Student,McKinleyville,95519,Murray Rd & Bates Rd,13.89,10,32,0.40,0.1,0.10,0.40,0.00,0.00,Drive Alone,False,True,4.61,332


### Cities should be consistent in naming

In [337]:
df['city'] = df['city'].str.title().str.strip()

In [338]:
df.city.unique()

array(['Blue Lake', 'Fortuna', 'Trinidad', 'Eureka', 'Mckinleyville',
       'Arcata', nan], dtype=object)

## Messy Population Data Cleaning

In [339]:
df2 = pd.read_csv("../raw_data/population_addresses_2024_25_messy_100.csv")
df2.head()

,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
0,POP_S_7315039,Student,7917 J St,Eureka,California,955,6863 Cedar Ln,Redding,CA,96001.0,NorCal
1,POP_S_4194026,Student,1845 Bates Rd,McKinleyville,CA,95519,8942 Birch Pl,La Mesa,CA,91942.0,SanDiego
2,POP_S_4478952,Student,NaN,Eureka,CA,95501,9527 Maple Dr,Redding,CA,96001.0,NorCal
3,POP_S_2541649,Student,NaN,Eureka,CA,95501,1318 Meadow Blvd,National City,california,91950.0,SanDiego
4,POP_S_4662231,Student,8466 H St,Eureka,NV,NaN,3406 Meadow Pl,National City,CA,91950.0,SanDiego


In [340]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   pop_id            100 non-null    object 
 1   role              100 non-null    object 
 2   current_street    94 non-null     object 
 3   current_city      96 non-null     object 
 4   current_state     98 non-null     object 
 5   current_zip       97 non-null     object 
 6   permanent_street  100 non-null    object 
 7   permanent_city    100 non-null    object 
 8   permanent_state   100 non-null    object 
 9   permanent_zip     98 non-null     float64
 10  permanent_region  100 non-null    object 
dtypes: float64(1), object(10)
memory usage: 8.7+ KB


### Address invalid current_zip entries

In [342]:
null_zip2 = df2[df2['current_zip'].notna() == False]
null_zip2.loc[:, 'current_zip'] = '00000'
df2.update(null_zip2)

df2.loc[df2.current_zip == 'ABCDE', 'current_zip'] = '00000'
df2['current_zip'] = df2['current_zip'].astype(str).str.ljust(5, fillchar='0').astype(float).astype(int)
df2.head()

,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
0,POP_S_7315039,Student,7917 J St,Eureka,California,95500,6863 Cedar Ln,Redding,CA,96001.0,NorCal
1,POP_S_4194026,Student,1845 Bates Rd,McKinleyville,CA,95519,8942 Birch Pl,La Mesa,CA,91942.0,SanDiego
2,POP_S_4478952,Student,NaN,Eureka,CA,95501,9527 Maple Dr,Redding,CA,96001.0,NorCal
3,POP_S_2541649,Student,NaN,Eureka,CA,95501,1318 Meadow Blvd,National City,california,91950.0,SanDiego
4,POP_S_4662231,Student,8466 H St,Eureka,NV,0,3406 Meadow Pl,National City,CA,91950.0,SanDiego


In [343]:
df2.current_zip.unique()

array([95500, 95519, 95501,     0, 99999, 95521, 95540, 95525, 95570])

### Same with permanent_zip

In [358]:
df2.loc[df2.permanent_zip.isna(), 'permanent_zip'] = 0
df2['permanent_zip'] = df2['permanent_zip'].astype(int)
df2.head()

,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
0,POP_S_7315039,Student,7917 J St,Eureka,CA,95500,6863 Cedar Ln,Redding,CA,96001,NorCal
1,POP_S_4194026,Student,1845 Bates Rd,Mckinleyville,CA,95519,8942 Birch Pl,La Mesa,CA,91942,SanDiego
2,POP_S_4478952,Student,No Answer,Eureka,CA,95501,9527 Maple Dr,Redding,CA,96001,NorCal
3,POP_S_2541649,Student,No Answer,Eureka,CA,95501,1318 Meadow Blvd,National City,california,91950,SanDiego
4,POP_S_4662231,Student,8466 H St,Eureka,NV,0,3406 Meadow Pl,National City,CA,91950,SanDiego


### Look at current_city unique values and acknowledge null entries

In [345]:
df2['current_city'] = df2['current_city'].str.title().str.strip()

In [ ]:
df2.loc[df2.current_city.isna(), 'current_city'] = 'No Answer'
df2.loc[df2.current_city == 'Arcatra', 'current_city'] = 'Arcata'
df2.loc[df2.current_city == 'Fake City', 'current_city'] = 'No Answer'
df2.loc[df2.current_city == 'Mckinelyville', 'current_city'] = 'Mckinleyville'
df2.loc[df2.current_city == 'Eurika', 'current_city'] = 'Eureka'

In [349]:
df2.current_city.unique()

array(['Eureka', 'Mckinleyville', 'No Answer', 'Arcata', 'Fortuna',
       'Blue Lake', 'Trinidad'], dtype=object)

### Same with current_state field

In [ ]:
df2.loc[df2.current_state.isna(), 'current_state'] = 'No Answer'
df2.loc[df2.current_state == 'California', 'current_state'] = 'CA'
df2.loc[df2.current_state == 'california', 'current_state'] = 'CA'
df2.loc[df2.current_state == 'ca', 'current_state'] = 'CA'
df2.loc[df2.current_state == 'XX', 'current_state'] = 'No Answer'

In [352]:
df2.current_state.unique()

array(['CA', 'NV', 'No Answer', 'WA'], dtype=object)

### Same with current_street

In [354]:
df2.loc[df2.current_street.isna(), 'current_street'] = 'No Answer'

In [356]:
df2.current_street.unique()

array(['7917 J St', '1845 Bates Rd', 'No Answer', '8466 H St',
       '9999 Nonexistent Ln', '4132 Broadway St', '8337 G St',
       '6727 K St', '7492 Heartwood Dr', '7836 Central Ave',
       '1675 spears rd', '9522 Hiller Rd', '5038 broadway st', '442 G St',
       '9115 Alliance Rd', '6600 Alliance Rd', '4446 Q St',
       '7456 Baldwin St', '5161 LK Wood Blvd', '6767 Wabash Ave',
       '7959 murray rd', '9678 14th St', '2260 LK Wood Blvd',
       '3473 6th St', '2051 baldwin st', '4540 Main St',
       '7341 Alliance Rd', '6002 6th St', '3534 Broadway St',
       '215 Taylor Way', '9007 11th St', '1257 Broadway St',
       '8485 Hiller Rd', '2555 South Rail St', '9250 Alliance Rd',
       '7059 Main St', '1611 Harris St', '6998 11th St',
       '2208 Baldwin St', '163 K St', '9252 Spears Rd', '4641 Baldwin St',
       '7778 LK Wood Blvd', '9141 H St', '3077 Samoa Blvd', '5674 5th st',
       '9297 11th St', '1097 14th St', '6574 Broadway St',
       '4586 Spears Rd', '5615 Allian

### Make permanent_city title case and permanent_state corrected

In [360]:
df2.permanent_city = df2.permanent_city.str.title().str.strip()
df2.loc[df2.permanent_state == 'california', 'permanent_state'] = 'CA'

### Final look at both dataframes before geocode-correcting address entries, zipcodes, states, cities, and distances to campus

In [ ]:
df.to_csv("../clean_data/Clean_Survey_Data.csv", index=False)

In [365]:
df.head(10)

,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est
0,F_2021-22_00104,2021,Faculty,Blue Lake,95500,South Rail St & Hatchery Rd,9.63,9,36,0.44,0.0,0.11,0.11,0.11,0.22,Drive Alone,False,False,0.58,325
1,F_2024-25_00271,2024,Faculty,Fortuna,95540,Main St & Kenmar Rd,20.56,9,37,0.72,0.0,0.00,0.14,0.00,0.14,Drive Alone,False,False,0.99,349
2,S_2024-25_00154,2024,Staff/Admin,Trinidad,95570,Patrick Point Dr & Trinity St,11.82,10,48,0.80,0.1,0.00,0.00,0.00,0.10,Drive Alone,False,False,2.98,486
3,S_2024-25_00197,2024,Student,Eureka,95501,I St & K St,8.61,10,32,0.50,0.1,0.10,0.20,0.00,0.10,Drive Alone,False,False,2.73,315
4,S_2022-23_00708,2022,Student,Mckinleyville,95519,Murray Rd & Bates Rd,13.89,10,32,0.40,0.1,0.10,0.40,0.00,0.00,Drive Alone,False,True,4.61,332
5,F_2020-21_00048,2020,Faculty,Arcata,95521,14th St & Alliance Rd,0.49,6,34,0.17,0.0,0.00,0.17,0.17,0.50,Telecommute,False,True,0.04,192
6,S_2023-24_00496,2023,Student,NaN,0,Trinity St & Edwards St,10.44,10,32,0.30,0.3,0.10,0.30,0.00,0.00,Drive Alone,False,False,2.72,326
7,S_2022-23_00188,2022,Student,Eureka,95501,K St & Harris St,10.20,10,30,0.30,0.1,0.20,0.20,0.20,0.00,Drive Alone,False,False,2.19,297
8,S_2021-22_00554,2021,Student,Arcata,95521,Union St & H St,1.37,9,30,0.38,0.0,0.24,0.27,0.05,0.05,Walk,False,False,0.35,258
9,S_2020-21_00177,2020,Student,Arcata,95521,Samoa Blvd & G St,2.37,5,28,0.42,0.0,0.26,0.21,0.00,0.11,Walk,False,False,0.12,134


In [366]:
df2.to_csv("../clean_data/Clean_Population_Addresses.csv", index=False)

In [367]:
df2.head(10)

,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
0,POP_S_7315039,Student,7917 J St,Eureka,CA,95500,6863 Cedar Ln,Redding,CA,96001,NorCal
1,POP_S_4194026,Student,1845 Bates Rd,Mckinleyville,CA,95519,8942 Birch Pl,La Mesa,CA,91942,SanDiego
2,POP_S_4478952,Student,No Answer,Eureka,CA,95501,9527 Maple Dr,Redding,CA,96001,NorCal
3,POP_S_2541649,Student,No Answer,Eureka,CA,95501,1318 Meadow Blvd,National City,CA,91950,SanDiego
4,POP_S_4662231,Student,8466 H St,Eureka,NV,0,3406 Meadow Pl,National City,CA,91950,SanDiego
5,POP_S_9756042,Student,9999 Nonexistent Ln,No Answer,No Answer,99999,9913 Cedar Blvd,Glendale,CA,91203,LA
6,POP_S_4654016,Student,4132 Broadway St,Eureka,CA,95501,4568 Birch Blvd,Redding,CA,96001,NorCal
7,POP_S_2112110,Student,8337 G St,Arcata,CA,95521,7500 Cedar Pl,Pasadena,CA,91101,LA
8,POP_S_6276986,Student,6727 K St,Eureka,CA,95501,2426 Canyon Ln,Los Angeles,CA,90026,LA
9,POP_S_8719581,Student,7492 Heartwood Dr,Mckinleyville,CA,95519,8462 Canyon Ave,Glendale,CA,91203,LA


## Google Geocoding, Reverse Geocoding, and Distance Calculations using Python Client for Google Maps

In [ ]:
survey_df = pd.read_csv("../clean_data/Clean_Survey_Data.csv")

In [421]:
pop_df = pd.read_csv("../clean_data/Clean_Population_Addresses.csv")

### Function to geocode addresses from survey data, overwrite with correct values

In [ ]:
import os
import googlemaps
from datetime import datetime
import dotenv

dotenv.load_dotenv('.env')

gmaps = googlemaps.Client(key=os.getenv('API_KEY'))

import math

def _s(v):
    if v is None: return ""
    if isinstance(v, float):
        if math.isnan(v): return ""
        if v.is_integer(): return str(int(v))
    return str(v)

def _comp(ac, *kinds):
    for c in ac:
        ts = set(c.get("types", []))
        if any(k in ts for k in kinds):
            return c.get("long_name")
    return None

_gc = {}

def geocode_fill_row(row, inter_col="nearest_intersection", city_col="city", state_col="state", zip_col="zip",
                     country="US", default_state="CA"):
    q_inter = _s(row.get(inter_col)).strip()
    city_in = _s(row.get(city_col)).strip()
    state_in = _s(row.get(state_col)).strip() or default_state
    zip_in = _s(row.get(zip_col)).strip()

    comps = {"country": country, "administrative_area_level_1": state_in}
    if city_in: comps["locality"] = city_in
    if zip_in:  comps["postal_code"] = zip_in

    query = ", ".join(x for x in [q_inter or None, city_in or None, state_in, country] if x)

    if not query:
        row[state_col] = state_in or default_state
        return row

    cache_key = (query, tuple(sorted(comps.items())))
    if cache_key in _gc:
        res = _gc[cache_key]
    else:
        try:
            res = gmaps.geocode(address=query, components=comps)
            if not res:
                res = gmaps.geocode(query)
        except Exception:
            res = []
        _gc[cache_key] = res

    if not res:
        row[state_col] = state_in or default_state
        return row

    res.sort(key=lambda r: r.get("geometry", {}).get("location_type") != "ROOFTOP")
    g = res[0]
    ac = g["address_components"]

    row[city_col]  = _comp(ac, "locality") or _comp(ac, "postal_town") or _comp(ac, "sublocality", "sublocality_level_1") or city_in
    row[zip_col]   = _comp(ac, "postal_code") or zip_in
    row[state_col] = _comp(ac, "administrative_area_level_1") or state_in or default_state

    loc = g["geometry"]["location"]
    row["lat"], row["lng"] = loc["lat"], loc["lng"]
    return row



In [ ]:
for c in ("city", "zip", "state"):
    if c in survey_df.columns:
        survey_df[c] = survey_df[c].astype("string")

survey_df = survey_df.apply(
    lambda r: geocode_fill_row(r, inter_col="nearest_intersection", city_col="city", state_col="state", zip_col="zip"),
    axis=1
)

In [414]:
survey_df.head()

,respondent_id,year,role,city,zip,nearest_intersection,distance_to_campus_miles,trips_per_week,weeks_per_year,share_drive,share_carpool,share_bus,share_walk,share_bike,share_tele,mode_primary,is_ev_user,is_ebike_user,mtcde_est,total_trips_est,state,lat,lng
0,F_2021-22_00104,2021,Faculty,Blue Lake,95500,South Rail St & Hatchery Rd,9.63,9,36,0.44,0.0,0.11,0.11,0.11,0.22,Drive Alone,False,False,0.58,325,California,40.879070,-123.991850
1,F_2024-25_00271,2024,Faculty,Fortuna,95540,Main St & Kenmar Rd,20.56,9,37,0.72,0.0,0.00,0.14,0.00,0.14,Drive Alone,False,False,0.99,349,California,40.577488,-124.139811
2,S_2024-25_00154,2024,Staff/Admin,Trinidad,95570,Patrick Point Dr & Trinity St,11.82,10,48,0.80,0.1,0.00,0.00,0.00,0.10,Drive Alone,False,False,2.98,486,California,41.059999,-124.143143
3,S_2024-25_00197,2024,Student,Eureka,95501,I St & K St,8.61,10,32,0.50,0.1,0.10,0.20,0.00,0.10,Drive Alone,False,False,2.73,315,California,40.793227,-124.158087
4,S_2022-23_00708,2022,Student,McKinleyville,95519,Murray Rd & Bates Rd,13.89,10,32,0.40,0.1,0.10,0.40,0.00,0.00,Drive Alone,False,True,4.61,332,California,40.956505,-124.088442


In [415]:
survey_df.to_csv("../clean_data/Geocoded_Survey_Data.csv", index=False)

### Function to geocode addresses from population dataframe, overwrite with correct values from current or permanent address fields and flag sources of working addresses

In [422]:
import re, math

# --- keep your existing helpers (_s, _comp, _gc) ---

def _is_missing(v):
    s = _s(v).strip().lower()
    return (not s) or s in {"na", "n/a", "none", "no answer", "null"}

def _bad_zip(z):
    s = _s(z).strip()
    if s in {"", "0", "00000", "99999", "95500"}:
        return True
    return not re.match(r"^\d{5}(-\d{4})?$", s)

def _pick_best(res):
    if not res: return None
    res.sort(key=lambda r: r.get("geometry", {}).get("location_type") != "ROOFTOP")
    return res[0]

def _geocode_address(street, city=None, state=None, postal=None, country="US", default_state="CA"):
    street = _s(street).strip()
    city   = _s(city).strip() or None
    state  = (_s(state).strip() or default_state)
    postal = _s(postal).strip() or None
    if not street or _is_missing(street):
        return None, state

    comps = {"country": country, "administrative_area_level_1": state}
    if city:   comps["locality"] = city
    if postal and not _bad_zip(postal): comps["postal_code"] = postal

    q = ", ".join(x for x in [street, city or None, state, postal or None, country] if x)

    cache_key = ("addr", street.lower(), (city or "").lower(), state, (postal or ""), country)
    if cache_key in _gc:
        res = _gc[cache_key]
    else:
        try:
            res = gmaps.geocode(address=street, components=comps)
            if not res:
                res = gmaps.geocode(q)
        except Exception:
            res = []
        _gc[cache_key] = res

    return _pick_best(res), state

def _apply_geo_to_row(row, mapping, g, fallback_state="CA"):
    """
    mapping: {'street': 'current_street', 'city':'current_city', 'state':'current_state', 'postal':'current_zip'}
    Writes back in-place. Returns (row, ok_bool_with_valid_zip)
    """
    if g is None:
        # ensure state isn’t blank
        if mapping.get('state'):
            row[mapping['state']] = _s(row.get(mapping['state'])).strip() or fallback_state
        return row, False

    ac = g["address_components"]

    def comp(*kinds):
        for c in ac:
            ts = set(c.get("types", []))
            if any(k in ts for k in kinds):
                return c.get("long_name")
        return None

    # Street (street_number + route)
    if mapping.get('street'):
        sn = (comp("street_number") or "").strip()
        rt = (comp("route") or "").strip()
        base = _s(row.get(mapping['street'])).strip()
        row[mapping['street']] = (sn + " " + rt).strip() or base

    # City with fallbacks
    if mapping.get('city'):
        row[mapping['city']] = comp("locality") or comp("postal_town") or comp("sublocality", "sublocality_level_1") or _s(row.get(mapping['city'])).strip()

    # State (prefer geocode; else fallback_state)
    if mapping.get('state'):
        row[mapping['state']] = comp("administrative_area_level_1") or _s(row.get(mapping['state'])).strip() or fallback_state

    # ZIP (prefer geocode; if still bad, keep original)
    ok_zip = False
    if mapping.get('postal'):
        new_zip = comp("postal_code") or _s(row.get(mapping['postal'])).strip()
        row[mapping['postal']] = new_zip
        ok_zip = not _bad_zip(new_zip)

    # lat/lng
    loc = g["geometry"]["location"]
    row["lat"], row["lng"] = loc["lat"], loc["lng"]

    # success = got a usable ZIP OR at least a valid geocode
    return row, ok_zip

def _correct_row(row, mapping, country="US", default_state="CA"):
    g, norm_state = _geocode_address(
        street=row.get(mapping.get('street')),
        city=row.get(mapping.get('city')),
        state=row.get(mapping.get('state')) or default_state,
        postal=row.get(mapping.get('postal')),
        country=country, default_state=default_state
    )
    return _apply_geo_to_row(row, mapping, g, fallback_state=norm_state or default_state)

def correct_row_with_fallback(row, current_map, perm_map=None, country="US", default_state="CA"):
    """
    Try CURRENT first; if it fails or produces a bad ZIP, try PERMANENT.
    """
    # If current street is missing / "No Answer", skip straight to permanent (if available)
    cur_street = _s(row.get(current_map.get('street'))).strip()
    current_unusable = _is_missing(cur_street)

    if not current_unusable:
        row, ok = _correct_row(row, current_map, country, default_state)
    else:
        ok = False

    if ok or not perm_map:
        row["geocode_source"] = "current" if ok and not current_unusable else None
        return row

    # Only fall back if permanent street exists and isn’t "No Answer"
    perm_street = _s(row.get(perm_map.get('street'))).strip()
    if _is_missing(perm_street):
        row["geocode_source"] = "current" if ok else None
        return row

    row2, ok2 = _correct_row(row, perm_map, country, default_state)
    if ok2:
        row2["geocode_source"] = "permanent"
        return row2

    row["geocode_source"] = "current" if ok else None
    return row


In [423]:
# make sure these are strings so writes stick (no 0/NaN weirdness)
for c in ("current_street","current_city","current_state","current_zip",
          "permanent_street","permanent_city","permanent_state","permanent_zip"):
    if c in pop_df.columns:
        pop_df[c] = pop_df[c].astype("string")

pop_df = pop_df.apply(
    lambda r: correct_row_with_fallback(
        r,
        current_map={"street":"current_street","city":"current_city","state":"current_state","postal":"current_zip"},
        perm_map   ={"street":"permanent_street","city":"permanent_city","state":"permanent_state","postal":"permanent_zip"},
        country="US", default_state="CA"
    ),
    axis=1
)

In [425]:
pop_df.head()

,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region,lat,lng,geocode_source
0,POP_S_7315039,Student,J Street,Eureka,California,95500,6863 Cedars Road,Redding,California,96001,NorCal,40.528818,-122.383587,permanent
1,POP_S_4194026,Student,1845 Bates Road,McKinleyville,California,95519,8942 Birch Pl,La Mesa,CA,91942,SanDiego,40.948300,-124.102645,current
2,POP_S_4478952,Student,No Answer,Eureka,CA,95501,2510 Airpark Drive,Redding,California,96001,NorCal,40.570782,-122.401749,permanent
3,POP_S_2541649,Student,No Answer,Eureka,CA,95501,1318 Meadow Drive,National City,California,91950,SanDiego,32.677624,-117.079066,permanent
4,POP_S_4662231,Student,H Street,Eureka,California,0,Meadow Drive,National City,California,91950,SanDiego,32.677194,-117.079443,permanent


In [426]:
pop_df.to_csv("../clean_data/Geocoded_Population_Addresses.csv", index=False)

## Done with Data Pipeline